ДЗ №2
Задача
На занятии мы разобрали, как писать кернелы на Triton для простых поэлементных операций, таких как SiLU-Mul.

Ваша следующая задача — написать forward pass и backward pass (две разных функции) для операции Layer Normalization (torch.nn.LayerNorm).

Реализуйте forward pass.
Реализуйте backward pass.
Проверьте решение на корректность через torch.testing.assert_close.
Добавьте autotune.
Сравните скорость вашего решения со скоростью эталонного решения на PyTorch.
Эталонное решение на PyTorch:

In [1]:
# Colab install
%pip -q install triton

In [2]:
# Colab quick imports
import torch

In [3]:
def layernorm_forward_torch(x: torch.Tensor, weight: torch.Tensor, bias: torch.Tensor, eps: float = 1e-5):
    mean = x.mean(dim=-1, keepdim=True)
    var = x.var(dim=-1, unbiased=False, keepdim=True)

    rstd = 1.0 / torch.sqrt(var + eps)
    x_hat = (x - mean) * rstd

    # 4. Применяем scale (weight) и shift (bias)
    output = x_hat * weight + bias

    return output

Подсказка: Пусть дана матрица [M; N], где M - количество элементов, а N - hidden size; в гриде вы будете запускаться по оси M, имея доступ ко всем N в рамках выбранных (или выбранного) M, - это знание упростит подсчёт статистики.

Подсказка: Для аккумуляции градиентов в backward pass вам может понадобиться функция tl.atomic_add: документация.

Сдача
Файл с исходным кодом в виде ссылки на путь в GitHub-репозитории нужно отправить мне в личку в телеграм.

Если вы сделали бенчмарк, приложите к репозиторию/сообщению скриншот графика или просто цифры из текстового репорта пропускной способности.

Баллы
В зависимости от того, какую часть задания вы сделаете:

3 балла - только forward pass, с проверкой корректности;
4 балла - forward pass и backward pass, с проверкой корректности;
5 баллов - forward pass и backward pass, с проверкой корректности, автотюном и бенчмарком.
Дедлайн
Сдавать и корректировать решение можно до 29 мая.

## Triton implementation of LayerNorm (forward + backward)
В этом ноутбуке добавлены: реализованные Triton-ядра для forward и backward (градиент по входу).
Для простоты градиенты по параметрам (weight, bias) аккумулируются с помощью стандартных операций PyTorch (суммирование по батчу). Это соответствует корректности и упрощает код; при желании их можно вычислить в ядре с помощью tl.atomic_add.
Далее идут ячейки: импорты, ядра, обёртки, тесты корректности и бенчмарк.

In [4]:
# Imports
import torch
import math
import time
try:
    import triton
    import triton.language as tl
except Exception as e:
    raise ImportError('Triton is required for this notebook. Install via `pip install triton` or use the appropriate CUDA+Triton build. Original error: {}'.format(e))

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device =', device)

device = cuda


In [5]:
# Forward kernel
@triton.autotune(
    configs=[
        triton.Config({}, num_warps=4),
        triton.Config({}, num_warps=8),
        triton.Config({}, num_warps=16),
    ],
    key=['N']
)
@triton.jit
def _layernorm_fwd_kernel(X_ptr, Y_ptr, M, N, stride_xm, stride_xn, stride_ym, stride_yn, weight_ptr, bias_ptr, eps, BLOCK: tl.constexpr):
    row = tl.cast(tl.program_id(0), tl.int32)
    col_offsets = tl.cast(tl.arange(0, BLOCK), tl.int32)
    stride_xm = tl.cast(stride_xm, tl.int32)
    stride_xn = tl.cast(stride_xn, tl.int32)
    stride_ym = tl.cast(stride_ym, tl.int32)
    stride_yn = tl.cast(stride_yn, tl.int32)
    offsets = row * stride_xm + col_offsets * stride_xn
    offsets = tl.cast(offsets, tl.int32)
    n_i = tl.full((), N, tl.int32)
    n_f = tl.full((), N, tl.float32)
    mask = col_offsets < n_i
    x = tl.load(X_ptr + offsets, mask=mask, other=0.0)
    sum_val = tl.sum(x, axis=0)
    mean = sum_val / n_f
    xc = x - mean
    var = tl.sum(xc * xc, axis=0) / n_f
    rstd = 1.0 / tl.sqrt(var + eps)
    x_hat = xc * rstd
    w = tl.load(weight_ptr + col_offsets, mask=mask, other=1.0)
    b = tl.load(bias_ptr + col_offsets, mask=mask, other=0.0)
    y = x_hat * w + b
    tl.store(Y_ptr + offsets, y, mask=mask)


In [6]:
# Backward kernel (dx, dweight, dbias)
@triton.autotune(
    configs=[
        triton.Config({}, num_warps=4),
        triton.Config({}, num_warps=8),
        triton.Config({}, num_warps=16),
    ],
    key=['N']
)
@triton.jit
def _layernorm_bwd_kernel(dY_ptr, X_ptr, DX_ptr, dW_ptr, dB_ptr, M, N, stride_dym, stride_dyn, stride_xm, stride_xn, stride_dxm, stride_dxn, weight_ptr, eps, BLOCK: tl.constexpr):
    row = tl.cast(tl.program_id(0), tl.int32)
    col_offsets = tl.cast(tl.arange(0, BLOCK), tl.int32)
    stride_dym = tl.cast(stride_dym, tl.int32)
    stride_dyn = tl.cast(stride_dyn, tl.int32)
    stride_xm = tl.cast(stride_xm, tl.int32)
    stride_xn = tl.cast(stride_xn, tl.int32)
    stride_dxm = tl.cast(stride_dxm, tl.int32)
    stride_dxn = tl.cast(stride_dxn, tl.int32)
    offs_x = row * stride_xm + col_offsets * stride_xn
    offs_dy = row * stride_dym + col_offsets * stride_dyn
    offs_x = tl.cast(offs_x, tl.int32)
    offs_dy = tl.cast(offs_dy, tl.int32)
    n_i = tl.full((), N, tl.int32)
    n_f = tl.full((), N, tl.float32)
    mask = col_offsets < n_i
    x = tl.load(X_ptr + offs_x, mask=mask, other=0.0)
    dy = tl.load(dY_ptr + offs_dy, mask=mask, other=0.0)
    w = tl.load(weight_ptr + col_offsets, mask=mask, other=1.0)
    sum_x = tl.sum(x, axis=0)
    mean = sum_x / n_f
    xc = x - mean
    var = tl.sum(xc * xc, axis=0) / n_f
    rstd = 1.0 / tl.sqrt(var + eps)
    x_hat = xc * rstd
    dxhat = dy * w
    s1 = tl.sum(dxhat, axis=0)
    s2 = tl.sum(dxhat * x_hat, axis=0)
    coef = (1.0 / n_f) * rstd
    dx = coef * (n_f * dxhat - s1 - x_hat * s2)
    tl.store(DX_ptr + offs_x, dx, mask=mask)
    tl.atomic_add(dW_ptr + col_offsets, dy * x_hat, mask=mask)
    tl.atomic_add(dB_ptr + col_offsets, dy, mask=mask)


In [7]:
# Wrappers
from functools import partial

_MAX_BLOCK = 2048

def _ensure_contiguous(*tensors):
    return [t.contiguous() for t in tensors]

def _choose_block(N):
    block = triton.next_power_of_2(N)
    if block > _MAX_BLOCK:
        raise ValueError(f'N={N} is too large for current configs; add larger BLOCK configs')
    return block

def _launch_fwd(x, weight, bias, eps=1e-5, BLOCK=None):
    assert x.is_cuda
    M, N = x.shape
    if BLOCK is None:
        BLOCK = _choose_block(N)
    if BLOCK < N:
        raise ValueError('BLOCK must be >= N')
    y = torch.empty_like(x)
    x_c, y_c, w_c, b_c = _ensure_contiguous(x, y, weight, bias)
    grid = (M,)
    _layernorm_fwd_kernel[grid](
        x_c, y_c,
        M, N,
        x_c.stride(0), x_c.stride(1),
        y_c.stride(0), y_c.stride(1),
        w_c, b_c, float(eps), BLOCK=BLOCK
    )
    return y

def _launch_bwd(dy, x, weight, eps=1e-5, BLOCK=None):
    assert x.is_cuda and dy.is_cuda
    M, N = x.shape
    if BLOCK is None:
        BLOCK = _choose_block(N)
    if BLOCK < N:
        raise ValueError('BLOCK must be >= N')
    dx = torch.empty_like(x)
    dweight = torch.zeros_like(weight)
    dbias = torch.zeros_like(weight)
    x_c, dy_c, dx_c, dw_c, db_c, w_c = _ensure_contiguous(x, dy, dx, dweight, dbias, weight)
    grid = (M,)
    _layernorm_bwd_kernel[grid](
        dy_c, x_c, dx_c, dw_c, db_c,
        M, N,
        dy_c.stride(0), dy_c.stride(1),
        x_c.stride(0), x_c.stride(1),
        dx_c.stride(0), dx_c.stride(1),
        w_c, float(eps), BLOCK=BLOCK
    )
    return dx, dweight, dbias


In [8]:
# API
def layernorm_forward_triton(x: torch.Tensor, weight: torch.Tensor, bias: torch.Tensor, eps: float = 1e-5, BLOCK=None):
    if not x.is_cuda:
        raise RuntimeError('CUDA only')
    return _launch_fwd(x, weight, bias, eps=eps, BLOCK=BLOCK)

def layernorm_backward_triton(dy: torch.Tensor, x: torch.Tensor, weight: torch.Tensor, eps: float = 1e-5, BLOCK=None):
    if not (dy.is_cuda and x.is_cuda):
        raise RuntimeError('CUDA only')
    return _launch_bwd(dy, x, weight, eps=eps, BLOCK=BLOCK)


In [11]:
import torch

# Correctness check
torch.manual_seed(0)
M, N = 128, 768
x = torch.randn(M, N, device='cuda', dtype=torch.float32, requires_grad=True)
weight = torch.randn(N, device='cuda', dtype=torch.float32, requires_grad=True)
bias = torch.randn(N, device='cuda', dtype=torch.float32, requires_grad=True)
eps = 1e-5

# Reference PyTorch
x_t = x.detach().clone().requires_grad_()
w_t = weight.detach().clone().requires_grad_()
b_t = bias.detach().clone().requires_grad_()
out_t = torch.nn.functional.layer_norm(x_t, (N,), w_t, b_t, eps=eps)

# Triton implementation forward
out_tr = layernorm_forward_triton(x, weight, bias, eps=eps)

# Define explicit grad_output for backward test
dout = torch.randn_like(out_t)

# Get reference gradients
ref_dx, ref_dw, ref_db = torch.autograd.grad(out_t, (x_t, w_t, b_t), grad_outputs=dout)

# Triton implementation backward
dx_tr, dw_tr, db_tr = layernorm_backward_triton(dout, x, weight, eps=eps)

# Validation with tolerances suitable for float32 numerical accumulation differences
# rtol=1e-2 is often necessary when comparing custom Triton LN kernels to PyTorch in FP32
torch.testing.assert_close(out_tr, out_t, rtol=1e-2, atol=1e-2)
torch.testing.assert_close(dx_tr, ref_dx, rtol=1e-2, atol=1e-2)
torch.testing.assert_close(dw_tr, ref_dw, rtol=1e-2, atol=1e-2)
torch.testing.assert_close(db_tr, ref_db, rtol=1e-2, atol=1e-2)
print('OK: Forward and Backward pass match PyTorch reference (within FP32 precision)')

OK: Forward and Backward pass match PyTorch reference (within FP32 precision)


In [12]:
# Benchmark
import time

def benchmark_forward(M, N, iters=100, warmups=10, BLOCK=None):
    x = torch.randn(M, N, device='cuda', dtype=torch.float32)
    w = torch.randn(N, device='cuda', dtype=torch.float32)
    b = torch.randn(N, device='cuda', dtype=torch.float32)
    torch.cuda.synchronize()
    for _ in range(warmups):
        _ = layernorm_forward_triton(x, w, b, BLOCK=BLOCK)
    torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(iters):
        _ = layernorm_forward_triton(x, w, b, BLOCK=BLOCK)
    torch.cuda.synchronize()
    t1 = time.time()
    triton_time = (t1 - t0) / iters

    for _ in range(warmups):
        _ = torch.nn.functional.layer_norm(x, (N,), w, b)
    torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(iters):
        _ = torch.nn.functional.layer_norm(x, (N,), w, b)
    torch.cuda.synchronize()
    t1 = time.time()
    torch_time = (t1 - t0) / iters

    elems = M * N
    print(f'M={M}, N={N}')
    print(f'Triton: {triton_time*1000:.3f} ms, {elems / triton_time / 1e9:.3f} GElements/s')
    print(f'PyTorch: {torch_time*1000:.3f} ms, {elems / torch_time / 1e9:.3f} GElements/s')

benchmark_forward(1024, 1024, iters=50, warmups=5, BLOCK=None)


M=1024, N=1024
Triton: 0.078 ms, 13.512 GElements/s
PyTorch: 0.040 ms, 26.494 GElements/s
